# NMF on ModulAir 00688

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00688
data = pd.read_csv('/content/MOD-00688.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T07:32:44Z,577143173,2025-12-31T02:32:44Z,MOD-00688,44.0,-2.3,0.983,0.108,0.042,0.008,0.010,...,35.320,38.781,14337.0,14338.0,14339.0,14475.0,14500.0,14550.0,14525.0,3.41
2025-12-31T07:31:44Z,577143170,2025-12-31T02:31:44Z,MOD-00688,44.0,-2.3,1.003,0.086,0.034,0.008,0.003,...,35.320,39.128,14337.0,14338.0,14339.0,14475.0,14500.0,14550.0,14525.0,2.29
2025-12-31T07:30:44Z,577143172,2025-12-31T02:30:44Z,MOD-00688,44.0,-2.3,1.187,0.079,0.021,0.005,0.008,...,35.556,38.781,14337.0,14338.0,14339.0,14475.0,14500.0,14550.0,14525.0,2.25
2025-12-31T07:29:44Z,577143171,2025-12-31T02:29:44Z,MOD-00688,43.6,-2.3,0.912,0.083,0.043,0.019,0.005,...,34.863,40.180,14337.0,14338.0,14339.0,14475.0,14500.0,14550.0,14525.0,2.89
2025-12-31T07:28:44Z,577141355,2025-12-31T02:28:44Z,MOD-00688,43.5,-2.3,1.069,0.099,0.030,0.006,0.003,...,34.630,40.183,14337.0,14338.0,14339.0,14475.0,14500.0,14550.0,14525.0,4.20


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T07:32:44Z,2025-12-31T02:32:44Z,743.444,1.892,35.320,38.781,0.983,0.108,0.042,0.008,0.010,0.008
2025-12-31T07:31:44Z,2025-12-31T02:31:44Z,740.981,1.892,35.320,39.128,1.003,0.086,0.034,0.008,0.003,0.005
2025-12-31T07:30:44Z,2025-12-31T02:30:44Z,726.613,1.892,35.556,38.781,1.187,0.079,0.021,0.005,0.008,0.000
2025-12-31T07:29:44Z,2025-12-31T02:29:44Z,698.983,1.891,34.863,40.180,0.912,0.083,0.043,0.019,0.005,0.000
2025-12-31T07:28:44Z,2025-12-31T02:28:44Z,674.834,1.891,34.630,40.183,1.069,0.099,0.030,0.006,0.003,0.003


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T02:32:44Z,743.444,1.892,35.320,38.781,0.983,0.108,0.042,0.008,0.010,0.008
1,2025-12-31T02:31:44Z,740.981,1.892,35.320,39.128,1.003,0.086,0.034,0.008,0.003,0.005
2,2025-12-31T02:30:44Z,726.613,1.892,35.556,38.781,1.187,0.079,0.021,0.005,0.008,0.000
3,2025-12-31T02:29:44Z,698.983,1.891,34.863,40.180,0.912,0.083,0.043,0.019,0.005,0.000
4,2025-12-31T02:28:44Z,674.834,1.891,34.630,40.183,1.069,0.099,0.030,0.006,0.003,0.003


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 02:32:44,743.444,1.892,35.320,38.781,0.983,0.108,0.042,0.008,0.010,0.008
1,2025-12-31 02:31:44,740.981,1.892,35.320,39.128,1.003,0.086,0.034,0.008,0.003,0.005
2,2025-12-31 02:30:44,726.613,1.892,35.556,38.781,1.187,0.079,0.021,0.005,0.008,0.000
3,2025-12-31 02:29:44,698.983,1.891,34.863,40.180,0.912,0.083,0.043,0.019,0.005,0.000
4,2025-12-31 02:28:44,674.834,1.891,34.630,40.183,1.069,0.099,0.030,0.006,0.003,0.003


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,921.90,34.06,35.17,1.86,17.40,2.07,0.92,0.37,0.53,0.39
1,2025-03-31 21:00:00,901.35,28.61,37.18,2.61,22.81,4.18,1.33,0.44,0.61,0.49
2,2025-03-31 22:00:00,982.99,33.25,35.50,2.84,21.84,5.83,1.61,0.47,0.58,0.40
3,2025-03-31 23:00:00,738.15,15.45,56.75,2.76,9.03,1.99,0.58,0.16,0.19,0.12
4,2025-04-01 00:00:00,687.43,20.48,57.53,2.76,10.59,2.21,0.62,0.17,0.20,0.13
...,...,...,...,...,...,...,...,...,...,...,...
5888,2025-12-30 22:00:00,675.55,35.31,40.20,1.78,0.99,0.12,0.04,0.01,0.01,0.01
5889,2025-12-30 23:00:00,678.02,35.38,39.62,1.84,1.02,0.11,0.04,0.01,0.01,0.00
5890,2025-12-31 00:00:00,674.12,35.36,39.18,1.85,1.15,0.15,0.05,0.01,0.01,0.01
5891,2025-12-31 01:00:00,665.45,35.74,38.21,1.87,1.06,0.11,0.04,0.01,0.01,0.00


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,921.90,34.06,35.17,1.86,17.40,2.07,0.92,0.37,0.53,0.39
2025-03-31 21:00:00,901.35,28.61,37.18,2.61,22.81,4.18,1.33,0.44,0.61,0.49
2025-03-31 22:00:00,982.99,33.25,35.50,2.84,21.84,5.83,1.61,0.47,0.58,0.40
2025-03-31 23:00:00,738.15,15.45,56.75,2.76,9.03,1.99,0.58,0.16,0.19,0.12
2025-04-01 00:00:00,687.43,20.48,57.53,2.76,10.59,2.21,0.62,0.17,0.20,0.13


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.487293,0.473713,0.425632,0.031461,0.255770,0.076895,0.068148,0.082960,0.136247,0.314516
2025-03-31 21:00:00,0.476431,0.397914,0.449958,0.044147,0.335293,0.155275,0.098519,0.098655,0.156812,0.395161
2025-03-31 22:00:00,0.519584,0.462448,0.429626,0.048038,0.321035,0.216568,0.119259,0.105381,0.149100,0.322581
2025-03-31 23:00:00,0.390167,0.214882,0.686797,0.046685,0.132736,0.073923,0.042963,0.035874,0.048843,0.096774
2025-04-01 00:00:00,0.363358,0.284840,0.696236,0.046685,0.155667,0.082095,0.045926,0.038117,0.051414,0.104839


In [11]:
data.to_csv('MOD-00688_timeseries_hourly_scaled.csv')

## Implementing NMF

In [12]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [13]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.484608,0.476254,0.433229,0.048670,0.213802,0.175583,0.134596,0.114341,0.125236,0.192345
2025-03-31 21:00:00,0.492993,0.393216,0.451322,0.048007,0.287715,0.236452,0.180074,0.152114,0.164652,0.246172
2025-03-31 22:00:00,0.523086,0.462304,0.432352,0.051368,0.303969,0.236378,0.176103,0.147215,0.158277,0.237737
2025-03-31 23:00:00,0.414751,0.204559,0.677613,0.042176,0.121616,0.069387,0.049402,0.040974,0.050939,0.089877
2025-04-01 00:00:00,0.433990,0.254468,0.668928,0.044305,0.123303,0.073881,0.053530,0.044773,0.054959,0.096145
...,...,...,...,...,...,...,...,...,...,...
2025-12-30 19:00:00,0.401351,0.494935,0.453443,0.043466,0.018421,0.000000,0.000000,0.000961,0.007051,0.028629
2025-12-30 20:00:00,0.399993,0.484967,0.460649,0.043281,0.018714,0.000000,0.000000,0.000959,0.007147,0.028764
2025-12-30 21:00:00,0.400096,0.481711,0.464332,0.043276,0.018864,0.000000,0.000000,0.000959,0.007197,0.028866


In [14]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.037878,0.053050,0.108573,0.104629
1,0.036727,0.072836,0.084562,0.139312
2,0.033991,0.077439,0.102970,0.134006
3,0.068195,0.025440,0.038721,0.036256
4,0.067184,0.025991,0.051518,0.039851
...,...,...,...,...
5734,0.048282,0.000000,0.119876,0.000000
5735,0.049050,0.000000,0.117182,0.000000
5736,0.049442,0.000000,0.116280,0.000000
5737,0.049844,0.000000,0.113837,0.000000


In [15]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [16]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,4.135628,0.622675,9.391479,0.427316,0.381536,0.000000,0.000000,0.010186,0.136891,0.420380
Feature 2,2.441765,0.138638,1.460878,0.222498,3.757795,1.210723,0.391422,0.111630,0.000000,0.000000
Feature 3,1.682348,3.877928,0.000000,0.190480,0.000000,0.000000,0.000000,0.003917,0.003687,0.069505
Feature 4,0.150696,0.232023,0.000000,0.000000,0.000000,1.064286,1.087949,1.028471,1.143568,1.614045


In [17]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.037878,0.053050,0.108573,0.104629
2025-03-31 21:00:00,0.036727,0.072836,0.084562,0.139312
2025-03-31 22:00:00,0.033991,0.077439,0.102970,0.134006
2025-03-31 23:00:00,0.068195,0.025440,0.038721,0.036256
2025-04-01 00:00:00,0.067184,0.025991,0.051518,0.039851
...,...,...,...,...
2025-12-30 19:00:00,0.048282,0.000000,0.119876,0.000000
2025-12-30 20:00:00,0.049050,0.000000,0.117182,0.000000
2025-12-30 21:00:00,0.049442,0.000000,0.116280,0.000000


In [18]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,4.135628,0.622675,9.391479,0.427316,0.381536,0.000000,0.000000,0.010186,0.136891,0.420380
Factor 2,2.441765,0.138638,1.460878,0.222498,3.757795,1.210723,0.391422,0.111630,0.000000,0.000000
Factor 3,1.682348,3.877928,0.000000,0.190480,0.000000,0.000000,0.000000,0.003917,0.003687,0.069505
Factor 4,0.150696,0.232023,0.000000,0.000000,0.000000,1.064286,1.087949,1.028471,1.143568,1.614045


In [19]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.497234,0.174371,0.313198,0.006249,0.008948
no,0.495194,0.153145,0.341790,0.000000,0.009870
no2,0.092160,0.012187,0.888721,0.011844,-0.004913
o3,0.918173,0.084831,0.000000,0.000000,-0.003004
bin0,0.147913,0.865274,0.000000,0.000000,-0.013187
bin1,0.000000,0.752591,0.000000,0.384169,-0.136759
bin2,0.000000,0.414201,0.000000,0.668536,-0.082737
bin3,0.023413,0.152399,0.013941,0.815352,-0.005105
bin4,0.254949,0.000000,0.010632,0.734582,-0.000164
bin5,0.386592,0.000000,0.098972,0.511948,0.002489


In [20]:
results.to_csv('MOD-00688_factor_results.csv')
comp.to_csv('MOD-00688_factor_comp.csv')
res.to_csv('MOD-00688_factor_resid.csv')